In [ ]:
# %% Setup
from pathlib import Path
import sys, os, multiprocessing as mp
import torch

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

LOGICAL = mp.cpu_count() or 8
os.environ["OMP_NUM_THREADS"] = str(LOGICAL)
os.environ["MKL_NUM_THREADS"] = str(LOGICAL)
os.environ["OPENBLAS_NUM_THREADS"] = str(LOGICAL)
os.environ["NUMEXPR_NUM_THREADS"] = str(LOGICAL)

torch.set_num_threads(max(1, LOGICAL // 2))
torch.set_num_interop_threads(max(1, LOGICAL // 2))

try:
    import intel_extension_for_pytorch as ipex
    HAS_IPEX = True
    print("[info] Intel Extension for PyTorch (IPEX) detected.")
except Exception:
    HAS_IPEX = False
    ipex = None
    print("[info] IPEX not found; using stock PyTorch.")

from tools import model_selector, inference_options

print(f"[cpu] threads: {LOGICAL}")
print(f"[gpu] available: {torch.cuda.is_available()}")


In [ ]:
# %% Select input video
from tools import input_selector

VIDEO_EXTS = {".mp4", ".mov", ".mkv", ".avi", ".webm", ".m4v"}
video_path = input_selector.get_input_video_path(allowed_exts=VIDEO_EXTS)
if not video_path:
    raise SystemExit("No video selected.")
print(f"[ok] selected video: {video_path}")


In [ ]:
# %% Extract frames
from video_pipeline.frame_extraction import extract_frames

temp_root = ROOT / "temp"
temp_root.mkdir(parents=True, exist_ok=True)
frames_dir = extract_frames(Path(video_path), temp_root)
print(f"[ok] frames directory: {frames_dir}")


In [ ]:
# %% Optional grayscale contrast preprocessing
from PIL import Image, ImageEnhance

IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}

contrast_text = input("Pre-AI grayscale contrast multiplier (blank = 1.0, try 0.50 for noir): ").strip()
try:
    contrast_factor = float(contrast_text) if contrast_text else 1.0
except ValueError:
    contrast_factor = 1.0
    print("[warn] invalid contrast input, using 1.0")

frames_dir = Path(frames_dir)
if contrast_factor != 1.0:
    contrast_dir = frames_dir.parent / f"{frames_dir.name}_contrast_{int(contrast_factor * 100):03d}"
    contrast_dir.mkdir(parents=True, exist_ok=True)
    frame_paths = [p for p in sorted(frames_dir.iterdir()) if p.suffix.lower() in IMAGE_EXTS]
    for src in frame_paths:
        with Image.open(src) as img:
            gray = img.convert("L")
            adjusted = ImageEnhance.Contrast(gray).enhance(contrast_factor)
            adjusted.convert("RGB").save(contrast_dir / src.name)
    frames_dir = contrast_dir
    print(f"[ok] contrast-adjusted frames: {frames_dir}")
else:
    print("[info] contrast preprocessing skipped")


In [ ]:
# %% Choose model and colorize
from video_pipeline.colorization import run_colorization

available_models = model_selector.scan_available_models(ROOT / "tools" / "AImodels")
model_name = inference_options.choose_colorization_model(available_models)
model_options = inference_options.choose_chromanet_options(model_name)
print(f"[info] using model: {model_name}")

use_gpu = torch.cuda.is_available()
color_dir = run_colorization(Path(frames_dir), model_name, use_gpu, **model_options)
print(f"[ok] colorization complete: {color_dir}")


In [ ]:
# %% Temporal smoothing (optional)
from tools.TemporalSmoothing import apply_temporal_smoothing

def _normalize_path(p: str | Path) -> Path | None:
    if not p:
        return None
    p = str(p).strip().strip('"').strip("'").replace("\\", "/")
    return Path(p).expanduser().resolve()

raw_in = input("Input folder for smoothing (blank = last colorized output): ").strip()
input_folder = _normalize_path(raw_in) if raw_in else color_dir
if not input_folder or not Path(input_folder).exists():
    raise SystemExit("No valid folder supplied for smoothing.")

raw_window = input("Temporal smoothing window (blank = off, recommended = 9): ").strip()
smooth_dir = None
if raw_window:
    try:
        window_size = int(raw_window)
    except ValueError:
        window_size = 0
    if window_size % 2 == 0:
        window_size -= 1
    if window_size < 3:
        window_size = 3
    smooth_dir = input_folder.parent / f"{input_folder.name}_TemporalSmoothed"
    smooth_dir.mkdir(parents=True, exist_ok=True)
    apply_temporal_smoothing(
        input_folder=input_folder,
        output_folder=smooth_dir,
        use_onnx=True,
        window_size=window_size,
    )
    print(f"[ok] temporal smoothing complete: {smooth_dir}")
else:
    print("[info] temporal smoothing skipped; using colorized frames.")


In [ ]:
# %% Rebuild video
from tools.FFmpeg.rebuild_video import build_video_from_frames

frames_for_video = smooth_dir if smooth_dir and Path(smooth_dir).exists() else color_dir
raw_frames = input("Frames folder for video rebuild (blank = auto): ").strip()
frames_for_video = _normalize_path(raw_frames) if raw_frames else frames_for_video

if not frames_for_video or not Path(frames_for_video).exists():
    raise SystemExit("No valid frames directory for video rebuild.")

codec_choice = (input("Codec [h264/h265/av1] (default=h264): ").strip().lower() or "h264")
if codec_choice not in {"h264", "h265", "av1"}:
    print("[warn] Unsupported codec selection; using h264.")
    codec_choice = "h264"

fps_text = input("Frames per second (default=24): ").strip()
try:
    fps_value = int(fps_text) if fps_text else 24
except ValueError:
    fps_value = 24
    print("[warn] invalid FPS input, using 24.")

raw_output = input("Output video path (blank = auto next to frames): ").strip()
if raw_output:
    output_video = _normalize_path(raw_output)
else:
    suffix = ".mkv" if codec_choice == "av1" else ".mp4"
    frames_path_obj = Path(frames_for_video)
    output_video = frames_path_obj.parent / f"{frames_path_obj.name}{suffix}"

print(f"[info] rebuilding video from {frames_for_video} -> {output_video}")
build_video_from_frames(
    frames_dir=str(frames_for_video),
    output_path=str(output_video),
    codec=codec_choice,
    fps=fps_value,
    prefer_gpu=True,
    source_audio=str(video_path),
)
print(f"[ok] video saved: {output_video}")


In [ ]:
# %% (Optional) quick report
from video_pipeline.reporting import generate_report

generate_report(frames_gray_dir=frames_dir, frames_color_dir=color_dir)
print("[done] report written to reports/report.json")
